# 05. エンジン間の相互運用（Spark ⇄ Trino）

Iceberg はテーブルの形式（ファイルとメタデータの書き方）を定めた **オープンな仕様** です。
同じカタログ（Polaris）に接続していれば、Spark で書いたテーブルを Trino で読み書きすることも、その逆もできます。

このノートブックは **前半 → Trino の SQL → 後半** の順に進めます。

- テーブル: `handson.interop`
- `make up-all` で Spark と Trino の両方を起動しておく

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("05_interoperability").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 前半: Spark でテーブルを作って書き込む

In [ ]:
sql("DROP TABLE IF EXISTS handson.interop PURGE")
sql("CREATE TABLE handson.interop (trip_id BIGINT, vendor STRING, fare DECIMAL(10, 2)) USING iceberg")
sql("INSERT INTO handson.interop VALUES (1, 'A', 12.50), (2, 'B', 30.00), (3, 'A', 8.00)")
sql("SELECT * FROM handson.interop ORDER BY trip_id")

## Trino の SQL を実行する

ターミナルで次のコマンドを実行します。Trino が Spark の書いたテーブルを読み、追記・更新・列の追加を行います。

```sh
make trino-sql FILE=handson/05_interoperability/trino.sql
```

実行したら、後半に進みます。

## 後半: Trino の変更を Spark から読む

Spark のカタログはテーブルのメタデータを一時的にキャッシュしています。
他のエンジンが書き込んだ直後は、`REFRESH TABLE` で最新のメタデータを読み直します（キャッシュの有効期限は既定で 30 秒）。

In [ ]:
sql("REFRESH TABLE handson.interop")
sql("SELECT * FROM handson.interop ORDER BY trip_id")

Trino が追加した `note` 列も見えています。スキーマの変更もメタデータを通じて共有されるためです。

Spark からもう一度書き込んでみます。

In [ ]:
sql("UPDATE handson.interop SET note = 'updated by spark' WHERE trip_id = 1")
sql("SELECT * FROM handson.interop ORDER BY trip_id")

## 誰が書いたかを見る

スナップショットの `summary` には、書き込んだエンジンの名前（`engine-name`）とバージョンが記録されています。
Spark と Trino の書き込みが、1本の履歴として交互に積み重なっています。

In [ ]:
sql("""
SELECT committed_at, operation,
       summary['engine-name'] AS engine,
       summary['engine-version'] AS version,
       summary['added-records'] AS added_records
FROM handson.interop.snapshots
ORDER BY committed_at
""")

## まとめ

- Spark と Trino は同じ Polaris のカタログを通じて同じテーブルを読み書きでき、データのコピーは不要
- スキーマの変更も含め、変更はすべて Iceberg のメタデータを通じて共有される
- Spark 側はメタデータをキャッシュするので、他のエンジンの変更を直後に見るには `REFRESH TABLE` を使う
- スナップショットには書き込んだエンジンが記録される